In [3]:
import geopandas as gpd
import xarray as xr
from geodatasets import get_path

import xvec

In [7]:
counties = gpd.read_file(get_path("geoda.natregimes"))

In [8]:
counties

,REGIONS,NOSOUTH,POLY_ID,NAME,STATE_NAME,STATE_FIPS,CNTY_FIPS,FIPS,STFIPS,COFIPS,...,GI59,GI69,GI79,GI89,FH60,FH70,FH80,FH90,West,geometry
0,1.0,1.0,1,Lake of the Woods,Minnesota,27,077,27077,27,77,...,0.285235,0.372336,0.342104,0.336455,11.279621,5.400000,5.663881,9.515860,0,"POLYGON ((-95.34258 48.5467, -95.34081 48.7151..."
1,2.0,1.0,2,Ferry,Washington,53,019,53019,53,19,...,0.256158,0.360665,0.361928,0.360640,10.053476,2.600000,10.079576,11.397059,1,"POLYGON ((-118.8505 47.94969, -118.84732 48.47..."
2,2.0,1.0,3,Stevens,Washington,53,065,53065,53,65,...,0.283999,0.394083,0.357566,0.369942,9.258437,5.600000,6.812127,10.352015,1,"POLYGON ((-117.43777 48.04422, -117.54113 48.0..."
3,2.0,1.0,4,Okanogan,Washington,53,047,53047,53,47,...,0.258540,0.371218,0.381240,0.394519,9.039900,8.100000,10.084926,12.840340,1,"POLYGON ((-118.97096 47.93928, -118.97293 47.9..."
4,2.0,1.0,5,Pend Oreille,Washington,53,051,53051,53,51,...,0.243263,0.365614,0.358706,0.387848,8.243930,4.100000,7.557643,10.313002,1,"POLYGON ((-117.4375 49, -117.03098 49, -117.02..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3080,2.0,1.0,3081,La Paz,Arizona,04,012,04012,4,12,...,0.271556,0.364110,0.372662,0.405743,9.216414,8.000000,9.296093,12.379134,1,"POLYGON ((-114.51984 33.02767, -114.5583 33.03..."
3081,2.0,1.0,3082,Cibola,New Mexico,35,006,35006,35,6,...,0.287346,0.347426,0.363889,0.404823,9.645881,9.300000,11.925297,14.907666,1,"POLYGON ((-107.19484 34.58352, -107.7178 34.58..."
3082,0.0,0.0,3083,York,Virginia,51,199,51199,51,199,...,0.271201,0.279962,0.310047,0.326229,8.776244,6.300000,8.063281,9.825284,0,"POLYGON ((-76.39569 37.10771, -76.4027 37.0905..."
3083,0.0,0.0,3084,Prince William,Virginia,51,153,51153,51,153,...,0.240268,0.288488,0.289718,0.286416,7.525717,5.300000,8.626065,10.276895,0,"POLYGON ((-77.53178 38.56506, -77.72094 38.840..."


In [6]:
pop1960 = xr.DataArray(counties.PO60, coords=[counties.geometry], dims=["county"])
pop1960

<xarray.DataArray 'PO60' (county: 3085)> Size: 12kB
array([ 4304,  3889, 17884, ..., 21583, 50164, 39260],
      shape=(3085,), dtype=int32)
Coordinates:
  * county   (county) geometry 25kB POLYGON ((-95.34258270263672 48.546703338...

In [10]:
pop1960 = pop1960.xvec.set_geom_indexes("county", crs=counties.crs)
pop1960.xindexes

Indexes:
    county   GeometryIndex (crs=EPSG:4326)

In [11]:
population = xr.DataArray(
    counties[["PO60", "PO70", "PO80", "PO90"]],
    coords=(counties.geometry, [1960, 1970, 1980, 1990]),
    dims=("county", "year"),
).xvec.set_geom_indexes("county", crs=counties.crs)
population

<xarray.DataArray (county: 3085, year: 4)> Size: 49kB
array([[  4304,   3987,   3764,   4076],
       [  3889,   3655,   5811,   6295],
       [ 17884,  17405,  28979,  30948],
       ...,
       [ 21583,  33203,  44189,  53427],
       [ 50164, 111102, 166665, 250377],
       [ 39260,  43766,  55800,  65077]], shape=(3085, 4), dtype=int32)
Coordinates:
  * county   (county) geometry 25kB POLYGON ((-95.34258270263672 48.546703338...
  * year     (year) int64 32B 1960 1970 1980 1990
Indexes:
    county   GeometryIndex (crs=EPSG:4326)

In [12]:
cube = xr.Dataset(
    data_vars=dict(
        population=(["county", "year"], counties[["PO60", "PO70", "PO80", "PO90"]]),
        unemployment=(["county", "year"], counties[["UE60", "UE70", "UE80", "UE90"]]),
        divorce=(["county", "year"], counties[["DV60", "DV70", "DV80", "DV90"]]),
        age=(["county", "year"], counties[["MA60", "MA70", "MA80", "MA90"]]),
    ),
    coords=dict(county=counties.geometry, year=[1960, 1970, 1980, 1990]),
).xvec.set_geom_indexes("county", crs=counties.crs)
cube

<xarray.Dataset> Size: 370kB
Dimensions:       (county: 3085, year: 4)
Coordinates:
  * county        (county) geometry 25kB POLYGON ((-95.34258270263672 48.5467...
  * year          (year) int64 32B 1960 1970 1980 1990
Data variables:
    population    (county, year) int32 49kB 4304 3987 3764 ... 43766 55800 65077
    unemployment  (county, year) float64 99kB 7.9 9.0 5.903 ... 7.018 5.489
    divorce       (county, year) float64 99kB 1.859 2.62 3.747 ... 4.782 7.415
    age           (county, year) float64 99kB 28.8 30.5 34.5 ... 28.97 35.33
Indexes:
    county   GeometryIndex (crs=EPSG:4326)

In [13]:
cube.xvec.geom_coords

Coordinates:
  * county   (county) geometry 25kB POLYGON ((-95.34258270263672 48.546703338...

In [14]:
cube.xvec.geom_coords_indexed

Coordinates:
  * county   (county) geometry 25kB POLYGON ((-95.34258270263672 48.546703338...